In [1]:
import uuid
from typing import List, Dict 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Prepare Models

In [2]:
load_dotenv()

True

In [30]:
llm = ChatOpenAI(model="gpt-5.4-mini")
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

## Example Raw Docs

In [4]:
raw_texts = [
    """
    LangGraph is a framework for building stateful multi-agent systems.
    It extends traditional LLM application development by introducing
    explicit workflow graphs composed of nodes and edges. Each node
    performs a specific task, such as calling an LLM, executing a tool,
    retrieving documents, or performing custom business logic.

    One of LangGraph's most important features is its shared state.
    Instead of passing variables manually between functions, every node
    can read and update a centralized state object. This makes it easier
    to build long-running workflows, conversational agents, and systems
    that require memory across multiple reasoning steps.

    LangGraph also supports conditional routing through conditional
    edges. Rather than following a fixed sequence of operations, the
    workflow can dynamically choose the next node depending on the
    current state or the LLM's decision. This enables branching,
    retries, fallback logic, human approval steps, and error recovery.

    Tool execution is another important capability. An LLM can decide
    whether to call external tools such as web search, SQL databases,
    APIs, calculators, or Python functions. The tool results are then
    stored in the shared state and used during later reasoning steps.

    LangGraph is commonly used for AI agents, retrieval pipelines,
    autonomous workflows, planning systems, and production-grade
    multi-agent architectures where maintaining state and controlling
    execution flow are important.
    """,

    """
    Retrieval-Augmented Generation, commonly called RAG, combines
    information retrieval with large language models. Instead of relying
    only on the model's internal knowledge, RAG first retrieves relevant
    documents from an external knowledge base and supplies those
    documents as additional context during generation.

    A typical RAG pipeline begins by loading documents from PDFs,
    websites, databases, or other sources. These documents are split
    into smaller chunks before being converted into vector embeddings
    using an embedding model. The embeddings are stored inside a vector
    database such as FAISS, Chroma, Pinecone, or Milvus.

    When a user submits a query, the same embedding model converts the
    query into a vector representation. A similarity search identifies
    the most relevant document chunks, which are then passed into the
    language model as context. This allows the model to answer questions
    using current or private information that was unavailable during
    pretraining.

    Advanced RAG systems often include metadata filtering, hybrid search,
    re-ranking models, parent-child retrieval, contextual compression,
    and caching to improve both retrieval accuracy and response speed.
    RAG is widely used for enterprise chatbots, document assistants,
    question-answering systems, and knowledge management applications.
    """,

    """
    LoRA, which stands for Low-Rank Adaptation, is a parameter-efficient
    fine-tuning technique for large language models. Instead of updating
    every weight inside the neural network, LoRA freezes the original
    model parameters and trains only a small number of additional
    low-rank matrices.

    Because only a tiny fraction of the parameters are updated, LoRA
    dramatically reduces GPU memory usage, storage requirements, and
    training time. This allows developers to fine-tune models with
    relatively modest hardware while still achieving competitive
    performance on specialized tasks.

    After training, the adapter weights can either remain separate from
    the base model or be merged into the original model for deployment.
    Multiple LoRA adapters can also be created for different domains,
    allowing a single base model to support many specialized tasks
    without storing multiple complete copies of the model.

    LoRA is commonly used for domain adaptation, instruction tuning,
    chatbot customization, code generation, and document understanding.
    Compared with full fine-tuning, it is significantly more efficient
    while preserving most of the original model's capabilities. This
    makes LoRA one of the most widely adopted techniques for adapting
    modern large language models.
    """
]

In [5]:
documents = [
    Document(page_content=text, metadata={"doc_id": str(i)})
    for i, text in enumerate(raw_texts)
]

In [6]:
documents[0]

Document(metadata={'doc_id': '0'}, page_content="\n    LangGraph is a framework for building stateful multi-agent systems.\n    It extends traditional LLM application development by introducing\n    explicit workflow graphs composed of nodes and edges. Each node\n    performs a specific task, such as calling an LLM, executing a tool,\n    retrieving documents, or performing custom business logic.\n\n    One of LangGraph's most important features is its shared state.\n    Instead of passing variables manually between functions, every node\n    can read and update a centralized state object. This makes it easier\n    to build long-running workflows, conversational agents, and systems\n    that require memory across multiple reasoning steps.\n\n    LangGraph also supports conditional routing through conditional\n    edges. Rather than following a fixed sequence of operations, the\n    workflow can dynamically choose the next node depending on the\n    current state or the LLM's decision. 

## Function for Generating summaries 

In [22]:
def generate_summary(doc):
    """
    Generate a summary for each document 
    """
    prompt = f"""
    Generate a summary from the given text below:

    Provided Text:
    {doc}

    Rules: 
    1. Only return the summary without any addtional comments or extra words
    2. Text limit is within 300 words
    """.strip()
    
    return llm.invoke(prompt).content
    

In [24]:
generate_summary(raw_texts[0])

'LangGraph is a framework for building stateful multi-agent systems using explicit workflow graphs made of nodes and edges. Each node can perform tasks like calling an LLM, using tools, retrieving documents, or applying business logic. Its shared state lets nodes read and update a central memory, making it well suited for long-running workflows and multi-step conversational agents. Conditional edges enable dynamic routing, allowing branching, retries, fallback logic, human approval, and error recovery based on state or LLM decisions. LangGraph also supports external tool execution, with results stored in shared state for later use. It is commonly used for AI agents, retrieval pipelines, autonomous workflows, planning systems, and production-grade multi-agent architectures.'

## Split documents into detailed chunks


In [32]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

all_summaries = []
all_chunks = []

for doc in documents:
    
    parent_id = str(uuid.uuid4())

    # for summary vectors
    summary_data = Document(
        page_content= generate_summary(doc),
        metadata={
            "parent_id": parent_id,
            "source_doc_id": doc.metadata["doc_id"]
        }
    )

    # for chunk vectors
    chunks = splitter.split_documents([doc])

    for chunk in chunks: 
        chunk.metadata["parent_id"] = parent_id
        chunk.metadata["source_doc_id"] = doc.metadata["doc_id"]

    all_summaries.append(summary_data)
    all_chunks.extend(chunks)

In [33]:
all_summaries

[Document(metadata={'parent_id': '7e29c0c2-7773-4403-8366-80574ddb186c', 'source_doc_id': '0'}, page_content='LangGraph is a framework for building stateful multi-agent systems using explicit workflow graphs made of nodes and edges. Each node can perform tasks such as calling an LLM, using tools, retrieving documents, or running custom logic. Its shared state lets nodes read and update a central object, making it well suited for long-running workflows and agents that need memory across multiple steps. LangGraph also supports conditional routing, allowing workflows to branch dynamically based on state or LLM decisions, which enables retries, fallbacks, human approval, and error recovery. It further supports tool execution, where an LLM can call external tools like search, databases, APIs, calculators, or Python functions and store results for later use. It is commonly used for AI agents, retrieval pipelines, autonomous workflows, planning systems, and production-grade multi-agent archit

In [34]:
all_chunks

[Document(metadata={'doc_id': '0', 'parent_id': '7e29c0c2-7773-4403-8366-80574ddb186c', 'source_doc_id': '0'}, page_content='LangGraph is a framework for building stateful multi-agent systems.\n    It extends traditional LLM application development by introducing\n    explicit workflow graphs composed of nodes and edges. Each node\n    performs a specific task, such as calling an LLM, executing a tool,'),
 Document(metadata={'doc_id': '0', 'parent_id': '7e29c0c2-7773-4403-8366-80574ddb186c', 'source_doc_id': '0'}, page_content='retrieving documents, or performing custom business logic.'),
 Document(metadata={'doc_id': '0', 'parent_id': '7e29c0c2-7773-4403-8366-80574ddb186c', 'source_doc_id': '0'}, page_content="One of LangGraph's most important features is its shared state.\n    Instead of passing variables manually between functions, every node\n    can read and update a centralized state object. This makes it easier\n    to build long-running workflows, conversational agents, and sys

## Register summaries & chunks
Just creating in-memory vectorstores 

In [ ]:
summary_vectorstore = Chroma(
    collection_name="summary_index",
    embedding_function=embedding_model
)

summary_vectorstore.add_documents(all_summaries)


chunk_vectorstore = Chroma(
    collection_name="chunk_index",
    embedding_function=embedding_model
)

chunk_vectorstore.add_documents(all_chunks)

## Retrieve 

In [62]:
def hierarchical_retrieve(
    query: str,
    summary_k= 1, 
    chunk_k= 3,
):
    summary_results = summary_vectorstore.similarity_search(query=query, k=summary_k)
    summary_parent_ids = [
        summary_result.metadata.get("parent_id")
        for summary_result in summary_results
    ]

    retrieved_chunks = []
    for parent_id in summary_parent_ids:
        chunks = chunk_vectorstore.similarity_search(
            query=query,
            k=chunk_k,
            filter={"parent_id": parent_id}
        )

        retrieved_chunks.extend(chunks)

    return retrieved_chunks


In [63]:
hierarchical_retrieve("what is langgraph")

[Document(id='034ab7ed-75a8-456f-b6f8-e834d95f0455', metadata={'doc_id': '0', 'parent_id': '7e29c0c2-7773-4403-8366-80574ddb186c', 'source_doc_id': '0'}, page_content='LangGraph is a framework for building stateful multi-agent systems.\n    It extends traditional LLM application development by introducing\n    explicit workflow graphs composed of nodes and edges. Each node\n    performs a specific task, such as calling an LLM, executing a tool,'),
 Document(id='c3f47d37-e3e7-4069-89c8-2e301a657931', metadata={'parent_id': '7e29c0c2-7773-4403-8366-80574ddb186c', 'source_doc_id': '0', 'doc_id': '0'}, page_content="One of LangGraph's most important features is its shared state.\n    Instead of passing variables manually between functions, every node\n    can read and update a centralized state object. This makes it easier\n    to build long-running workflows, conversational agents, and systems"),
 Document(id='bbfb8679-39f0-4b99-a3a0-7e7a50514932', metadata={'doc_id': '0', 'parent_id': '7